# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Specify kwargs

In [ ]:
# Specify T variable
T_VAR = "T_3"

## should we use OHC for RO's 'h' variable?
USE_OHC = False

## should we subtract median?
SUBTRACT_MEDIAN = False

## specify MLD
MLD = 50

## specify entrainment depth (meters)
ENT_DEPTH_RANGE = slice(51, 80)

## MONTH FOR PLOTTING
SEL_MONTH = lambda x: x.sel(month=slice(10, 12)).mean("month")
# SEL_MONTH = lambda x: x.sel(month=slice(1,3)).mean("month")
# SEL_MONTH = lambda x: x.mean("month")
MONTH0 = 10

## WHETHER TO COMPUTE BY MEMBER
BY_MEMBER = True

if BY_MEMBER:
    DIMS = ["time"]
else:
    DIMS = ["time", "member"]

## Load pre-computed regression coefficients

In [ ]:
## get filepath
if SUBTRACT_MEDIAN:
    txt = "_zero_median"
else:
    txt = ""
if USE_OHC:
    suf = ""
else:
    suf = "_ssh"

## build filepath
bjerknes_fp = DATA_FP / f"bjerknes{txt}{suf}"

## path where data is located
save_fp = bjerknes_fp / f"coefs_xvars={T_VAR}-h_w"
load = lambda n: xr.open_dataset(save_fp / f"{n}.nc")
coefs = {n: load(n) for n in ["all", "pos", "neg"]}

save_fp = bjerknes_fp / f"coefs_xvars={T_VAR}"
load = lambda n: xr.open_dataset(save_fp / f"{n}.nc")
coefs_T = {n: load(n) for n in ["all", "pos", "neg"]}

## Funcs

### Misc

In [ ]:
def window(x, subtract_median=SUBTRACT_MEDIAN):
    """get data in windows"""
    x_windowed = src.utils.get_windowed(x, stride=120)

    ## handle median case
    if subtract_median:
        median = x_windowed.groupby("time.month").median(["time", "member"])
        x_windowed = x_windowed.groupby("time.month") - median

    return x_windowed


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


def regress_over_time(
    data_windowed,
    x_vars,
    y_vars,
    dims=["time", "member"],
):
    """regression over time"""

    ## shared args
    kwargs = dict(x_vars=x_vars, y_vars=y_vars, dims=dims)

    ## do regression
    coefs = data_windowed.groupby("time.month").map(src.utils.regress_xr_proj, **kwargs)

    return coefs


def regress_wrapper(data, x_vars, y_var, y_fn, dims=["time", "member"]):
    """regression over time"""

    ## prep data
    y_data = src.utils.reconstruct_wrapper(data[[f"{y_var}", f"{y_var}_comp"]], fn=y_fn)

    ## subset for data
    data_ = xr.merge([data[x_vars], y_data])

    return regress_over_time(data_, x_vars=x_vars, y_vars=[y_var], dims=dims)


def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1


def check_dims(x):
    """make sure dimensions are ok before averaging"""
    ## check if latitude is in ssh
    if "latitude" in x.dims:
        x_ = copy.deepcopy(x)
    else:
        x_ = x.expand_dims("latitude")

    ## check if z_t is in ssh
    if "z_t" in x.dims:
        x_ = copy.deepcopy(x_)
    else:
        x_ = x_.expand_dims("z_t")

    return x_


def get_ddt(data):
    """differentiate with respect to time"""
    data_ = copy.deepcopy(data)
    data_ = data_.assign_coords({"t_idx": ("time", np.arange(len(data_.time)))})
    data_ = data_.swap_dims({"time": "t_idx"})
    return data_.differentiate("t_idx").swap_dims({"t_idx": "time"})

### Plotting

In [ ]:
def make_scatter_ax(ax, anom_, xvar, yvar, month, label, by_season=True):
    """scatter plot of data for given month"""

    ## prep func
    if by_season:
        get_season = lambda x: src.utils.sel_month(
            x.resample({"time": "QS-JAN"}).mean(), month
        )

    else:
        get_season = lambda x: src.utils.sel_month(x, month)

    prep = lambda x: get_season(x).transpose("time", "member")

    ## get plot data
    plot_data = (prep(anom_[xvar]), prep(anom_[yvar]))

    ## get stats
    corr = xr.corr(*plot_data)
    cov = xr.cov(*plot_data)
    m = cov / plot_data[0].var()

    ## plot data
    ax.scatter(*plot_data, s=3, label=f"m = {m.item():.1e}\nr = {corr.item():.2f}")
    ax.set_title(f"{label}")

    ## formatting
    ax_kwargs = dict(ls="--", c="gray", lw=0.5)
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    ax.legend(prop=dict(size=10))

    return ax


def make_scatter(anom_, xvar, yvar, month, by_season=True):
    """scatter plot of data for given month"""

    fig, axs = plt.subplots(1, 4, figsize=(11, 2.5), layout="constrained")

    if "year" in anom_:
        for ax, year in zip(
            axs,
            [1870, 2010, 2050, 2080],
        ):

            ## scatter plot of data
            ax = make_scatter_ax(
                ax,
                anom_.sel(year=year, method="nearest"),
                xvar=xvar,
                yvar=yvar,
                month=month,
                by_season=by_season,
                label=year,
            )

    else:

        for ax, t_idx, label in zip(
            axs,
            [["1850", "1879"], ["1995", "2024"], ["2035", "2064"], ["2071", "2100"]],
            ["1865", "2010", "2050", "2085"],
        ):

            ## helper func
            prep = lambda x: x.sel(time=slice(*t_idx))

            ## scatter plot of data
            ax = make_scatter_ax(
                ax,
                prep(anom_),
                xvar=xvar,
                yvar=yvar,
                month=month,
                by_season=by_season,
                label=label,
            )

    ## format/scale axes
    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_timeseries(coefs, sel_fn=lambda x: x):
    """plot timeseries comparison"""

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5))

    ## loop thru pos and negative
    for i, (name, color) in enumerate(zip(["pos", "neg"], ["r", "b"])):

        ## plot median and bounds
        for q, lw in zip([0.5, 0.1, 0.9], [2, 0.6, 0.6]):

            ## plot neutral and pos/or neg
            for name_, color_ in zip(["all", name], ["k", color]):

                ## finally, plot data
                axs[i].plot(
                    coefs.year,
                    sel_fn(coefs)[name_].quantile(q=q, dim="member"),
                    c=color_,
                    lw=lw,
                )

            ## plot on shared axs
            axs[2].plot(
                coefs.year,
                sel_fn(coefs)[name].quantile(q=q, dim="member"),
                c=color,
                lw=lw,
            )

    ## formatting
    for ax in axs:
        ax.set_xticks([1870, 2010, 2080])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axvline(2010, **ax_kwargs)
        ax.axhline(0, **ax_kwargs)
    src.utils.set_lims(axs)

    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_zonal_structure(coefs, sel_fn=lambda x: x):
    """plot zonal structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ## select data
        for n, color in zip(["all", "pos", "neg"], ["k", "r", "b"]):

            ax.plot(coefs.longitude, coefs_y[n].mean("member"), c=color)

    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    for ax in axs:
        ax.set_xticks([140, 190, 240, 280])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(0, **ax_kwargs)

    return fig, axs


def add_vticks(axs, xticks, xlines=None):
    """add vertical lines to axs"""

    ## specify line style
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)

    ## loop thru axs
    for ax in axs:
        ax.set_xticks(xticks)
        if xlines is not None:
            for x0 in xlines:
                ax.axvline(x0, **ax_kwargs)

    return


def plot_vertical_structure(coefs, sel_fn):
    """plot vertical structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ## select data
        for n, color in zip(["all", "pos", "neg"], ["k", "r", "b"]):

            ax.plot(coefs_y[n].mean("member"), coefs.z_t, c=color)

    for ax in axs:
        ax.set_ylim([150, 5])

        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(50, **ax_kwargs)
        ax.axhline(80, **ax_kwargs)
        ax.axvline(0, **ax_kwargs)

    src.utils.set_lims(axs)
    axs[0].set_yticks([0, 50, 80])
    for ax in axs[1:]:
        ax.set_yticks([])
    return fig, axs

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### most data

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### "wide" subsurface data

In [ ]:
## should we use "wide" data?
USE_WIDE = True

## load spatial data
forced_wide, anom_wide = load_consolidated_wide()

if USE_WIDE:

    for v in list(forced_wide):
        forced[v] = forced_wide[v]
        anom[v] = anom_wide[v]

#### max grad thermocline

In [ ]:
h_mg_forced, h_mg = src.utils.load_h_data(max_grad=True)

### scaling for mean thermocline depth

#### Mean thermocline depth

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

### Compute OHC

In [ ]:
## should we use mixed layer T?
# use_T_ml = True
use_T_ml = False

## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
# LONS_W = dict(longitude=slice(120, 160))
LONS_W = dict(longitude=slice(120, 180))
LONS_TAU = dict(longitude=slice(150, 230))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
TAU_FN_3 = lambda x: sel_helper(x, LATS, LONS_E)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)


## specify entrainment / ML averages
LON_AVG = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")
LON_AVG_34 = lambda x: x.sel(longitude=slice(190, 240)).mean("longitude")
ENT_AVG = lambda x: x.sel(z_t=ENT_DEPTH_RANGE).mean("z_t")
ML_AVG = lambda x: x.sel(z_t=slice(None, MLD)).mean("z_t")

## get T3 volume avg
T3_ENT_AVG = lambda x: ENT_AVG(LON_AVG(x))
T3_ML_AVG = lambda x: ML_AVG(LON_AVG(x))
T34_ML_AVG = lambda x: ML_AVG(LON_AVG_34(x))

if use_T_ml:
    anom["T_3"] = src.utils.reconstruct_wrapper(
        anom[["T", "T_comp"]],
        fn=T3_ML_AVG,
    )["T"]

In [ ]:
lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

if USE_OHC:

    ## compute ohc
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W) / 300,
    )["T"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E) / 300,
    )["T"]

else:
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_W)),
    )["ssh"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_E)),
    )["ssh"]

## handle rectification

In [ ]:
## get variables to check
rect_vars = ["u", "w", "T", "sst", "taux", "nhf"]

## get total
total = forced[rect_vars] + anom[rect_vars]

## window it
total = window(total, subtract_median=False).groupby("time.month").mean()

## add sigma T info
total["sigma_T"] = window(Th["T_34"]).groupby("time.month").std()

## estimate rect. coefficients
rect_coefs = src.utils.estimate_rect_bymonth(total, xvar="sigma_T", yvars=rect_vars)

## get climatology
clim = total.mean("member")
clim_norect = rect_coefs.sel(degree=0)

## Analysis

### T vs. ddt(SST)$

#### Load coefficients

In [ ]:
## specify which longitudes to reduce over
if T_VAR == "T_34":
    ddt_LON_RANGE = [190, 240]

else:
    ddt_LON_RANGE = [210, 270]

## specify which variable to use ("T" or "sst")
YVAR = "sst"

## function to select data
sel_var = lambda x: x[f"ddt_{YVAR}"].sel(j=T_VAR)

## load coefs
R_proj = 12 * xr.merge(sel_var(c).rename(n) for n, c in coefs.items())

#### Plot spatial structure

reconstruct spatial strucure

In [ ]:
def reduce_yz(x):

    if "z_t" not in x.dims:
        x = x.expand_dims("z_t")

    if "latitude" not in x.dims:
        x = x.expand_dims("latitude")

    idx = dict(z_t=slice(None, MLD), latitude=slice(-5, 5))

    return x.sel(idx).mean(["z_t", "latitude"])


def reduce_x(x):

    ## get averaging dims
    idx = dict(longitude=slice(*ddt_LON_RANGE))

    return x.sel(idx).mean("longitude")


def recon_struct(coefs, v="T"):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename(v), anom[f"{v}_comp"]]),
        fn=reduce_yz,
    )

    return coefs_struct[v]


def recon_idx(coefs, v="T"):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename(v), anom[f"{v}_comp"]]),
        fn=lambda y: reduce_x(reduce_yz(y)),
    )

    return coefs_struct[v]

In [ ]:
## load coefs
R_struct = xr.merge(
    [recon_struct(R_proj[v], YVAR).rename(v) for v in ["all", "pos", "neg"]]
)

Plot

In [ ]:
fig, axs = plot_zonal_structure(R_struct, sel_fn=SEL_MONTH)
for ax in axs:
    ax.set_xlim([140, 280])

add_vticks(axs, xticks=[140, 190, 240, 280], xlines=[190, 240])
plt.show()

#### Timeseries

In [ ]:
## get index
R = reduce_x(R_struct)

fig, axs = plot_timeseries(coefs=R, sel_fn=SEL_MONTH)
plt.show()

#### Scatter plot

In [ ]:
scatter_data = xr.merge(
    [
        get_ddt(recon_idx(anom[YVAR], YVAR)).rename(f"ddt_{YVAR}"),
        anom[["T_3", "T_34", "h_w"]],
    ]
)

for xvar in [T_VAR, "h_w"]:
    fig, axs = make_scatter(scatter_data, xvar=xvar, yvar=f"ddt_{YVAR}", month=MONTH0)
    plt.show()

### Compare to RO

#### Get data

In [ ]:
## get data (remove median as well)
x = xr.merge([Th[T_VAR], anom["h_w"]])
# x /= x.std()
x = window(x)
y = get_ddt(x).rename({f"{v}": f"ddt_{v}" for v in list(x)})
z = xr.merge([x, y])

#### Version 1: piecewise linear

In [ ]:
## do regression
# z_pos = z.where(z[T_VAR]>0)

kwargs = dict(
    func=src.utils.regress_xr_proj,
    x_vars=list(x),
    y_vars=list(y),
    dims=["time", "member"],
)

regress_by_month = lambda x: 12 * x.groupby("time.month").map(**kwargs)

## find pos/neg
is_pos_ = z[T_VAR] > 0
is_neg_ = z[T_VAR] < 0

## do regression
coefs2_all = regress_by_month(z)
coefs2_pos = regress_by_month(z.where(is_pos_))
coefs2_neg = regress_by_month(z.where(is_neg_))
coefs2 = [coefs2_all, coefs2_pos, coefs2_neg]

## get R, epsilon
get_R = lambda x: x["ddt_T_3"].sel(j="T_3")
get_eps = lambda x: -x["ddt_h_w"].sel(j="h_w")
R2 = xr.merge([get_R(c).rename(n) for c, n in zip(coefs2, ["all", "pos", "neg"])])

Check reconstruction matches

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(8.5, 2.5), layout="constrained")
for ax, n in zip(axs, list(R)):
    ax.plot(R2.year, R2[n].mean("month"))
    ax.plot(R.year, R[n].mean(["month", "member"]), ls="--")
    ax.axhline(0, ls="--", c="k", lw=0.8)
    ax.set_title(n)

src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])

add_vticks(axs, xticks=[1870, 2010, 2090], xlines=[2010, 2090])
plt.show()

#### version 2: cubic

In [ ]:
import warnings


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits

In [ ]:
## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)
fit_kwargs = dict(ac_mask_idx=None, maskNT=["T2", "T3"], maskNH=[])

## get fits
fits = get_fits_over_time(
    x,
    model=MODEL,
    by_member=False,
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R3 = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps3 = -fits["Lac"].isel(ranky=1, rankx=1)
coefs3 = xr.concat([R3, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

## get coefficient array
# T0 = 1.5
sigma = x["T_3"].std().values.item()
w = sigma * np.linspace(-3, 3)
W = np.stack([w**i for i in range(3)], axis=1)
W = xr.DataArray(W, coords=dict(T=w, form=coefs3.form), dims=["T", "form"])

## reconstruct nonlinear R
R_nl = (coefs3 * W).sum("form")
R_nl_pos = R_nl.where(R_nl["T"] > 0).mean("T")
R_nl_neg = R_nl.where(R_nl["T"] < 0).mean("T")


# T0 = sigma * 1.5
T0 = 1.5
R_nl_pos2 = R_nl.sel(T=T0, method="nearest")
R_nl_neg2 = R_nl.sel(T=-T0, method="nearest")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(8.5, 2.5), layout="constrained")
for ax, n in zip(axs, list(R)):
    ax.plot(R2.year, R2[n].mean("month"))
    ax.plot(R.year, R[n].mean(["month", "member"]), ls="--")
    ax.axhline(0, ls="--", c="k", lw=0.8)
    ax.set_title(n)

axs[0].plot(params.year, params["R"].mean("cycle"), label=r"$R_0$")
axs[0].plot(params.year, R_nl.mean(["cycle", "T"]), label=r"avg($R$)")


axs[1].plot(params.year, R_nl_pos.mean("cycle"), c="gray")
axs[1].plot(params.year, R_nl_pos2.mean("cycle"), c="gray", ls="--")

axs[2].plot(params.year, R_nl_neg.mean("cycle"), c="gray")
axs[2].plot(params.year, R_nl_neg2.mean("cycle"), c="gray", ls="--")

axs[0].legend(prop=dict(size=8))
src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])

add_vticks(axs, xticks=[1870, 2010, 2090], xlines=[2010, 2050, 2090])
plt.show()

\begin{align}
    R &= R_0 + R_1T + R_2 T^2
\end{align}

In [ ]:
sel = lambda x: x.mean("cycle")  # .differentiate("year")
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(params.year, 0.2 * sel(params["R"]), c="k")
ax.plot(coefs_nl.year, sel(coefs_nl.sel(nro_form="T2")))
ax.plot(coefs_nl.year, sel(coefs_nl.sel(nro_form="T3")))
ax.axhline(0, ls="--", lw=0.8, c="gray")
ax.axvline(2050, ls="--", lw=0.8, c="gray")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2.5), layout="constrained")

ax.plot(params.year, R_nl.mean(["T", "cycle"]), c="k")
ax.plot(params.year, R_nl_pos.mean("cycle"), c="r")
ax.plot(params.year, R_nl_neg.mean("cycle"), c="b")

ax.plot(params.year, params["R"].mean("cycle"), c="k", ls="--")
ax.plot(params.year, R_nl_pos2.mean("cycle"), c="r", ls="--")
ax.plot(params.year, R_nl_neg2.mean("cycle"), c="b", ls="--")

add_vticks([ax], xticks=[1870, 2010, 2090], xlines=[2010, 2050])
plt.show()

color by magnitude of anomaly...

### T vs. Tsub

#### Load coefficients

In [ ]:
## load coefs
gamma_proj = xr.merge(c["T"].rename(n) for n, c in coefs.items())
# gamma_proj = xr.merge(c["T"].rename(n) for n, c in coefs_T.items())

#### Plot structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    # return x.sel(longitude=slice(190,240)).mean("longitude")
    return x.sel(longitude=slice(210, 270)).mean("longitude")


def reduce_z(x):
    """Get diff b/n Tsub and Tsurf, but don't average over longitude yet"""

    Tsurf = x.sel(z_t=slice(None, MLD)).mean("z_t")
    Tsub = x.sel(z_t=ENT_DEPTH_RANGE).mean("z_t")

    return Tsub - Tsurf


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("T"), anom["T_comp"]]),
        fn=reduce_z,
    )
    ## drop extra variables
    mu = coefs_["T"].squeeze(drop=True)

    return mu


def recon_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["T"], anom[f"T_comp"]]),
        fn=lambda y: reduce_x(reduce_z(y)),
    )

    return coefs_struct["T"]


def get_wsub(x):
    """Get upwelling at base of mixed layer"""

    wsub = x.sel(z_t=ENT_DEPTH_RANGE).mean("z_t")

    return wsub

In [ ]:
## get wbar
w_bar_struct = src.utils.reconstruct_wrapper(
    xr.merge([clim["w"], forced["w_comp"]]),
    fn=get_wsub,
)["w"]

## load coefs
gamma_struct = xr.merge(
    [recon_struct(gamma_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

# ## compute THF
# first, scale upwelling by mixed layer depth (50 m) and convert from m/mo to m/yr
w_H = 12 * w_bar_struct / MLD
thf_struct = gamma_struct * w_H

Make sure dimensions are ok

In [ ]:
if "j" not in list(gamma_struct.dims):
    gamma_struct = gamma_struct.expand_dims(j=[T_VAR])
    w_H = w_H.expand_dims(j=[T_VAR])
    thf_struct = thf_struct.expand_dims(j=[T_VAR])

Subsurface $T$

In [ ]:
for v in list(gamma_struct.j.values):

    fig, axs = plot_zonal_structure(coefs=gamma_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

thermocline feedback

In [ ]:
for v in gamma_struct.j.values:
    print(v)

    fig, axs = plot_zonal_structure(coefs=thf_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

#### Plot timeseries

In [ ]:
SEL_MONTH = lambda x: x.sel(month=slice(10, 12)).mean("month")
# SEL_MONTH = lambda x: x.mean("month")

## compute gamma
print("gamma")
gamma = reduce_x(gamma_struct)
fig, axs = plot_timeseries(coefs=gamma.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

## compute THF
print(f"\nTHF")
thf = reduce_x(thf_struct)
fig, axs = plot_timeseries(coefs=thf.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

print("wbar")
w_bar = reduce_x(w_bar_struct)
fig, axs = plot_timeseries(
    coefs=w_bar * xr.ones_like(gamma.sel(j=T_VAR)), sel_fn=SEL_MONTH
)
plt.show()

#### Scatter

In [ ]:
scatter_data = xr.merge(
    [
        recon_idx(anom[["T"]]),
        anom[["T_3", "T_34", "h_w"]],
    ]
)

for xvar in [T_VAR, "h_w"]:
    fig, axs = make_scatter(scatter_data, xvar=xvar, yvar=f"T", month=MONTH0)
    plt.show()

### Damping

#### Load data

In [ ]:
## get scaling factor to convert to damping rate (units of K/mo)
sec_per_mo = 8.64e4 * 30
rho = 1.02e3
Cp = 4.2e3
H0 = MLD
scale_fac = sec_per_mo / (rho * Cp * H0)

## helper func to get data
prep = lambda x: -12 * x["nhf"] * scale_fac

## load coefs
alpha_proj = xr.merge(prep(c).rename(n) for n, c in coefs_T.items())

#### structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    if T_VAR == "T_3":
        lon_range = slice(190, 240)
    else:
        lon_range = slice(210, 270)

    return x.sel(longitude=lon_range).mean("longitude")


def reduce_y(x):
    """Get diff b/n Tsub and Tsurf, but don't average over longitude yet"""

    return x.sel(latitude=slice(-5, 5)).mean("latitude")


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("nhf"), anom["nhf_comp"]]),
        fn=reduce_y,
    )
    ## drop extra variables
    coefs_ = coefs_["nhf"].squeeze(drop=True)

    return coefs_


def recon_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["nhf"], anom[f"nhf_comp"]]),
        fn=lambda z: reduce_x(reduce_y(z)),
    )

    return coefs_struct["nhf"]

In [ ]:
## load coefs
alpha_struct = xr.merge(
    [recon_struct(alpha_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

In [ ]:
fig, axs = plot_zonal_structure(coefs=alpha_struct, sel_fn=SEL_MONTH)
add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")
for ax in axs:
    ax.set_xlim([140, 280])
plt.show()

#### Timeseries

In [ ]:
alpha = reduce_x(alpha_struct)

fig, axs = plot_timeseries(coefs=alpha, sel_fn=SEL_MONTH)
plt.show()

#### Scatter

In [ ]:
scatter_data = xr.merge(
    [
        recon_idx(anom[["nhf"]]),
        anom[["T_3", "T_34"]],
    ]
)
fig, axs = make_scatter(scatter_data, xvar=T_VAR, yvar="nhf", month=MONTH0)
plt.show()

### wind stress

#### Load data

In [ ]:
## load coefs
mu_proj = xr.merge(c["taux"].rename(n) for n, c in coefs_T.items())

#### structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    return x.sel(longitude=slice(150, 270)).mean("longitude")


def reduce_y(x):
    """Get diff b/n Tsub and Tsurf, but don't average over longitude yet"""

    return x.sel(latitude=slice(-5, 5), longitude=slice(140, 280)).mean("latitude")


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("taux"), anom["taux_comp"]]),
        fn=reduce_y,
    )
    ## drop extra variables
    coefs_ = coefs_["taux"].squeeze(drop=True)

    return coefs_


def recon_taux_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["taux"], anom["taux_comp"]]),
        fn=lambda z: reduce_x(reduce_y(z)),
    )

    return coefs_struct["taux"]

In [ ]:
## load coefs
mu_struct = xr.merge(
    [recon_struct(mu_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

In [ ]:
fig, axs = plot_zonal_structure(coefs=mu_struct, sel_fn=SEL_MONTH)

add_vticks(axs, xticks=[150, 230, 270], xlines=[150, 230, 270])

plt.show()

#### Timeseries

In [ ]:
## compute index
mu = reduce_x(mu_struct)

fig, axs = plot_timeseries(coefs=mu, sel_fn=SEL_MONTH)
plt.show()

In [ ]:
scatter_data = xr.merge(
    [
        recon_taux_idx(anom[["taux"]]),
        anom[["T_3", "T_34"]],
    ]
)
fig, axs = make_scatter(scatter_data, xvar=T_VAR, yvar="taux", month=MONTH0)
plt.show()

#### Add index to ```anom```

In [ ]:
anom["taux_idx"] = recon_taux_idx(anom[["taux"]])

### Sverdrup balance

#### Compute

In [ ]:
def reduce_y(x):
    """reduce over lat/lon/depth"""

    ## make sure dims are all there
    x_ = check_dims(x)

    ## specify indices
    idx = dict(longitude=slice(120, 280), latitude=slice(-5, 5))

    ## average over lats
    x_ = x_.sel(idx).mean("latitude")

    ## integrate (optionally)
    if len(x_.z_t) > 1:
        x_ = x_.integrate("z_t") / 300
    else:
        x_ = x_.squeeze(drop=True)

    return x_


def postprocess_(coefs, T_var):
    """post-process regression coefficients"""

    ## get recon components
    comps = check_dims(anom[f"{T_var}_comp"])

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs, comps]),
        fn=reduce_y,
    )

    ## drop extra variables
    coefs_ = coefs_[T_var].squeeze(drop=True)

    return coefs_

In [ ]:
## specify wind variable (one of "T_34", "taux_idx")
x_var = "taux_idx"
# x_var = T_VAR

## specify h-variable (one of "T", "ssh")
if USE_OHC:
    y_var = "T"
else:
    y_var = "ssh"

## get data for regression (OHC)
beta_data = window(xr.merge([anom[y_var], anom[x_var]]))
regress_kwargs = dict(x_vars=[x_var], y_vars=[y_var], dims=DIMS)
regress_helper = lambda x: postprocess_(
    regress_over_time(x, **regress_kwargs), T_var=y_var
)

## do the regression
print("regress all")
beta_all = regress_helper(beta_data)
print("regress pos")
beta_pos = regress_helper(beta_data.where(beta_data[x_var] > 0))
print("regress neg")
beta_neg = regress_helper(beta_data.where(beta_data[x_var] < 0))

## merge into single dataarray
beta_struct = xr.merge(
    [beta_all.rename("all"), beta_pos.rename("pos"), beta_neg.rename("neg")]
)

#### spatial plot

In [ ]:
fig, axs = plot_zonal_structure(coefs=beta_struct, sel_fn=SEL_MONTH)

add_vticks(axs, xticks=[120, 170, 280], xlines=[170])

plt.show()

#### Scatter

##### Compute inices

In [ ]:
def get_he(x):
    """average over eastern Pacific"""

    ## lats/lons for averaging
    # idx = dict(longitude=slice(190, 240))
    idx = dict(longitude=slice(210, 270))

    return reduce_y(x).sel(idx).sum("longitude")


def get_hw(x):
    """average over western Pacific"""

    ## lats/lons for averaging
    idx = dict(longitude=slice(120, 170))

    return reduce_y(x).sel(idx).sum("longitude")

    return


def get_dh(x):
    """east-west difference"""

    return get_he(x) - get_hw(x)

In [ ]:
# add to dataarray
for n, fn in zip(["he", "hw", "dh"], [get_he, get_hw, get_dh]):
    beta_data[n] = src.utils.reconstruct_wrapper(
        xr.merge([beta_data[y_var], anom[f"{y_var}_comp"]]),
        fn=fn,
    )[y_var]

##### Make plot

In [ ]:
## specify kwargs
kwargs = dict(anom_=beta_data, xvar=x_var, month=MONTH0, by_season=True)

for yvar in ["dh", "he", "hw"]:
    print(yvar)

    fig, axs = make_scatter(yvar=yvar, **kwargs)
    plt.show()

#### Timeseries

In [ ]:
## compute beta index
# beta = get_he(beta_struct)
beta = get_dh(beta_struct)

fig, axs = plot_timeseries(coefs=beta, sel_fn=SEL_MONTH)
plt.show()

### Thermal expansion

#### Compute

In [ ]:
def reduce_z(x):
    """average over depth"""

    ## get indices
    idx = dict(longitude=slice(120, 280), z_t=ENT_DEPTH_RANGE)

    return x.sel(idx).mean("z_t")


def postprocess_Tsub(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs, anom["T_comp"]]),
        fn=reduce_z,
    )

    ## drop extra variables
    mu = coefs_["T"].squeeze(drop=True)

    return mu

In [ ]:
## get data for regression
ah_data = xr.merge([window(anom["T"]), beta_data["he"]])

In [ ]:
regress_kwargs = dict(x_vars=["he"], y_vars=["T"], dims=DIMS)
regress_helper = lambda x: postprocess_Tsub(regress_over_time(x, **regress_kwargs))

## do the regression
print("regress all")
ah_all = regress_helper(ah_data)
print("regress pos")
ah_pos = regress_helper(ah_data.where(ah_data["he"] > 0))
print("regress neg")
ah_neg = regress_helper(ah_data.where(ah_data["he"] < 0))

## merge into single dataarray
ah_struct = xr.merge([ah_all.rename("all"), ah_pos.rename("pos"), ah_neg.rename("neg")])

In [ ]:
fig, axs = plot_zonal_structure(coefs=ah_struct, sel_fn=SEL_MONTH)

add_vticks(axs, xticks=[190, 240], xlines=[190, 240])

plt.show()

In [ ]:
def reduce_x(T):
    ## specify averaging range
    idx = dict(longitude=slice(210, 270))

    ## compute
    return T.sel(idx).mean("longitude")


def get_Tsub_idx(T):
    """compute Tsub over central/east Pacific"""

    ## specify averaging range
    idx = dict(longitude=slice(210, 270))

    ## compute
    return reduce_x(reduce_z(T))


## add to dataarray
ah_data["Tsub"] = src.utils.reconstruct_wrapper(
    xr.merge([ah_data["T"], anom["T_comp"]]),
    fn=get_Tsub_idx,
)["T"]

In [ ]:
## specify kwargs
kwargs = dict(anom_=ah_data, xvar="he", yvar="Tsub", month=MONTH0, by_season=True)

fig, axs = make_scatter(**kwargs)
plt.show()

In [ ]:
## get index
ah = reduce_x(ah_struct)

fig, axs = plot_timeseries(coefs=ah, sel_fn=SEL_MONTH)
plt.show()

### zonal velocity

#### Load coefficients

In [ ]:
## load coefs
betau_proj = xr.merge(c["u"].rename(n) for n, c in coefs.items())

#### Plot structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    if T_VAR == "T_3":
        lon_range = [210, 270]
    else:
        lon_range = [190, 240]

    return x.sel(longitude=slice(*lon_range)).mean("longitude")


def reduce_z(x):
    """Average over ML don't average over longitude yet"""

    return x.sel(z_t=slice(None, MLD)).mean("z_t")


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("u"), anom["u_comp"]]),
        fn=reduce_z,
    )
    ## drop extra variables
    mu = coefs_["u"].squeeze(drop=True)

    return mu


def recon_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["u"], anom[f"u_comp"]]),
        fn=lambda y: reduce_x(reduce_z(y)),
    )

    return coefs_struct["u"]

In [ ]:
## load coefs
betau_struct = xr.merge(
    [recon_struct(betau_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

# ## Get zonal T profile
T_bar = src.utils.reconstruct_wrapper(
    # xr.merge([clim["T"], forced["T_comp"]]),
    xr.merge([clim_norect["T"], forced["T_comp"]]),
    fn=lambda x: reduce_z(x),
)["T"]

## compute zonal advective feedback
## mult. by 12 to convert from K/mo to K/yr
zaf_struct = 12 * -src.utils.get_udTdx(u=betau_struct, T=T_bar)

In [ ]:
for v in [T_VAR, "h_w"]:
    print(v)

    fig, axs = plot_zonal_structure(coefs=zaf_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    # axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

#### Plot timeseries

In [ ]:
## compute gamma
zaf = reduce_x(zaf_struct)

fig, axs = plot_timeseries(coefs=zaf.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

#### Scatter

In [ ]:
scatter_data = xr.merge(
    [
        recon_idx(anom[["u"]]),
        anom[["T_3", "T_34", "h_w"]],
    ]
)

for xvar in [T_VAR, "h_w"]:
    fig, axs = make_scatter(scatter_data, xvar=xvar, yvar="u", month=MONTH0)
    plt.show()

### Surface temp

#### Load coefficients

In [ ]:
## load coefs
betaT_proj = xr.merge(c["T"].rename(n) for n, c in coefs.items())

#### Plot structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    if T_VAR == "T_3":
        lon_range = [210, 270]
    else:
        lon_range = [190, 240]

    return x.sel(longitude=slice(*lon_range)).mean("longitude")


def reduce_x_discrete(x):
    """Get Tsub, but don't average over depth yet"""

    if T_VAR == "T_3":
        lon_range = [210, 270]
    else:
        lon_range = [190, 240]

    return x.sel(longitude=lon_range[1], method="nearest") - x.sel(
        longitude=lon_range[0], method="nearest"
    )


def reduce_z(x):
    """Average over ML don't average over longitude yet"""

    return x.sel(z_t=slice(None, MLD)).mean("z_t")


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("T"), anom["T_comp"]]),
        fn=reduce_z,
    )
    ## drop extra variables
    mu = coefs_["T"].squeeze(drop=True)

    return mu


def recon_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["T"], anom[f"T_comp"]]),
        fn=lambda y: reduce_x_discrete(reduce_z(y)),
    )

    return coefs_struct["T"]

In [ ]:
## load coefs
betaT_struct = xr.merge(
    [recon_struct(betaT_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

## Get u_bar
u_bar = src.utils.reconstruct_wrapper(
    # xr.merge([clim["u"], forced["u_comp"]]),
    xr.merge([clim_norect["u"], forced["u_comp"]]),
    fn=lambda x: reduce_z(x),
)["u"]

## compute dynamical damping
# multiply by 12 to convert from K/mo to K/yr
dd_struct = 12 * -src.utils.get_udTdx(u=u_bar, T=betaT_struct)

In [ ]:
for v in [T_VAR, "h_w"]:
    print(v)

    fig, axs = plot_zonal_structure(coefs=dd_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    # axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

#### Plot timeseries

In [ ]:
## compute gamma
dd = reduce_x(dd_struct)

fig, axs = plot_timeseries(coefs=dd.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

#### Scatter

In [ ]:
scatter_data = xr.merge(
    [
        recon_idx(anom[["T"]]),
        anom[["T_3", "T_34", "h_w"]],
    ]
)

for xvar in [T_VAR, "h_w"]:
    fig, axs = make_scatter(scatter_data, xvar=xvar, yvar="T", month=MONTH0)
    plt.show()

### NDH_x

In [ ]:
ndh_struct = 12 * -src.utils.get_udTdx(T=betaT_struct, u=betau_struct)

In [ ]:
for v in [T_VAR, "h_w"]:
    print(v)

    fig, axs = plot_zonal_structure(coefs=ndh_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    # axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

In [ ]:
## compute gamma
ndh = reduce_x(ndh_struct)

fig, axs = plot_timeseries(coefs=ndh.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

### sum of zonal

In [ ]:
zonal_struct = zaf_struct + dd_struct + ndh_struct
for v in [T_VAR, "h_w"]:
    print(v)

    fig, axs = plot_zonal_structure(coefs=zonal_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    # axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

In [ ]:
fig, axs = plot_timeseries(coefs=(zaf + dd + ndh).sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

### upwelling

#### Load coefficients

In [ ]:
## load coefs
betaw_proj = xr.merge(c["w"].rename(n) for n, c in coefs.items())

#### Plot structure

In [ ]:
def reduce_x(x):
    """Get Tsub, but don't average over depth yet"""

    if T_VAR == "T_3":
        lon_range = [210, 270]
    else:
        lon_range = [190, 240]

    # return x.sel(longitude=slice(190,240)).mean("longitude")
    return x.sel(longitude=slice(*lon_range)).mean("longitude")


def reduce_z(x):
    """average over entrainment zone"""

    return x.sel(z_t=ENT_DEPTH_RANGE).mean("z_t")


def recon_struct(coefs):
    """post-process regression coefficients"""

    ## reconstruct mean over equatorial region
    coefs_ = src.utils.reconstruct_wrapper(
        xr.merge([coefs.rename("w"), anom["w_comp"]]),
        fn=reduce_z,
    )
    ## drop extra variables
    mu = coefs_["w"].squeeze(drop=True)

    return mu


def recon_idx(coefs):
    """reconstruct spatial structure of coefficients"""

    coefs_struct = src.utils.reconstruct_wrapper(
        xr.merge([coefs["w"], anom[f"w_comp"]]),
        fn=lambda y: reduce_x(reduce_z(y)),
    )

    return coefs_struct["w"]


def get_dT(x):
    """Get diff b/n Tsub and Tsurf, but don't average over longitude yet"""

    Tsurf = x.sel(z_t=slice(None, MLD)).mean("z_t")
    Tsub = x.sel(z_t=ENT_DEPTH_RANGE).mean("z_t")

    return Tsub - Tsurf

In [ ]:
## load coefs
betaw_struct = xr.merge(
    [recon_struct(betaw_proj[v]).rename(v) for v in ["all", "pos", "neg"]]
)

## get dTbar
dT_bar_struct = src.utils.reconstruct_wrapper(
    xr.merge([clim["T"], forced["T_comp"]]),
    fn=get_dT,
)["T"]

## compute ekman
# first, scale upwelling by mixed layer depth (50 m) and convert from m/mo to m/yr
w_H = 12 * betaw_struct / MLD
ekf_struct = dT_bar_struct * w_H

In [ ]:
for v in [T_VAR, "h_w"]:
    print(v)

    fig, axs = plot_zonal_structure(coefs=ekf_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    for ax in axs:
        ax.set_xlim([240, 285])
    # axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

#### Plot timeseries

In [ ]:
## compute EKF
ekf = reduce_x(ekf_struct)

fig, axs = plot_timeseries(coefs=ekf.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

## compute gamma
dT_bar = reduce_x(dT_bar_struct)

fig, axs = plot_timeseries(
    coefs=xr.ones_like(ekf.sel(j=T_VAR)) * dT_bar, sel_fn=SEL_MONTH
)
plt.show()

#### Scatter

In [ ]:
scatter_data = xr.merge(
    [
        recon_idx(anom[["w"]]),
        anom[["T_3", "T_34", "h_w"]],
    ]
)

for xvar in [T_VAR, "h_w"]:
    fig, axs = make_scatter(scatter_data, xvar=xvar, yvar=f"w", month=MONTH0)
    plt.show()

### NDH_z

In [ ]:
w_H = betaw_struct / MLD
ndh_z_struct = 12 * gamma_struct * w_H

In [ ]:
for v in [T_VAR, "h_w"]:
    print(v)

    fig, axs = plot_zonal_structure(coefs=ndh_z_struct.sel(j=v), sel_fn=SEL_MONTH)
    add_vticks(axs, xticks=[190, 240, 280], xlines=[190, 240])
    # axs[0].set_ylabel(r"$T_{\text{sub}}~/~T-1$")

    plt.show()

In [ ]:
## compute gamma
ndh_z = reduce_x(ndh_z_struct)

fig, axs = plot_timeseries(coefs=ndh_z.sel(j=T_VAR), sel_fn=SEL_MONTH)
plt.show()

### Plot all vars

In [ ]:
def plot_timeseries_v2(ax, coefs, sel_fn=lambda x: x.mean("month"), norm=True):
    """plot timeseries comparison"""

    ## normalize coefficients
    coefs_ = sel_fn(coefs)
    coefs_norm = coefs_ / coefs_.isel(year=0).mean("member")["pos"].values.item()

    ## select coefficients to plot
    if norm:
        plot_coefs = coefs_norm
    else:
        plot_coefs = coefs_

    ## loop thru pos and negative
    for i, (name, color) in enumerate(zip(["pos", "neg", "all"], ["r", "b", "k"])):

        ## plot median and bounds
        if name == "all":
            q_vals = [0.5]
        else:
            q_vals = [0.5, 0.1, 0.9]

        for q, lw in zip(q_vals, [2, 0.6, 0.6]):

            ## finally, plot data
            ax.plot(
                plot_coefs.year,
                plot_coefs[name].quantile(q=q, dim="member"),
                c=color,
                lw=lw,
            )

    ## formatting
    ax.set_xticks([1870, 2010, 2080])
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)
    ax.axvline(2010, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    # ax.set_yticks([])

    return fig, axs

First, plot thermocline feedback estimates

In [ ]:
## specify which months to plot
kwargs = dict(sel_fn=lambda x: x.mean("month"))
# kwargs = dict(sel_fn = SEL_MONTH)
# kwargs = dict(sel_fn = lambda x : x.sel(month=3))

fig, axs = plt.subplots(1, 5, figsize=(10, 2.5), layout="constrained")

## mu
plot_timeseries_v2(axs[0], coefs=mu, **kwargs)

## beta
plot_timeseries_v2(axs[1], coefs=beta, **kwargs)

## THERMAL expansion
plot_timeseries_v2(axs[2], coefs=ah, **kwargs)

## product
plot_timeseries_v2(axs[3], coefs=mu * beta * ah - 1, norm=False, **kwargs)

## Direct estimate
plot_timeseries_v2(axs[4], coefs=gamma.isel(j=0), norm=False, **kwargs)


## label
labels = [
    r"$\mu$",
    r"$\beta$",
    r"$a_h$",
    r"$\mu\cdot \beta \cdot a_h-1$",
    "direct estimate",
]
for ax, label in zip(axs, labels):
    ax.set_title(label)

##formatting
src.utils.set_lims(axs[:3])
axs[0].set_yticks([0, 0.5, 1])

plt.show()

##### Plot growth rate stuff

Load epsilon data

In [ ]:
get_coef = lambda n, v: 12 * coefs[n][f"ddt_{v}"].sel(j=v).isel(mode=0).rename(n)

R_check = xr.merge(get_coef(n, T_VAR).rename(n) for n in list(coefs))
eps = xr.merge(-get_coef(n, "h_w").rename(n) for n in list(coefs))


get_F1 = lambda n: 12 * coefs[n][f"ddt_{T_VAR}"].sel(j="h_w").isel(mode=0).rename(n)
get_F2 = lambda n: -12 * coefs[n][f"ddt_h_w"].sel(j=T_VAR).isel(mode=0).rename(n)
F1 = xr.merge(get_F1(n).rename(n) for n in list(coefs))
F2 = xr.merge(get_F2(n).rename(n) for n in list(coefs))

Check R matches

In [ ]:
sel = lambda x: SEL_MONTH(x).mean("member")

## check R matches
fig, axs = plt.subplots(1, 3, figsize=(7.5, 2.5))
for ax, n in zip(axs, list(coefs)):
    ax.set_title(n)
    ax.plot(R.year, sel(R[n]), c="k")
    ax.plot(R.year, sel(R_check[n]), c="orange", ls="--")
    # ax.plot(R.year, sel(eps[n]), c="green")
    # ax.axvline(2030)


for ax in axs:
    ax.axhline(0, ls="--", c="k", lw=0.8)
for ax in axs[1:]:
    ax.set_yticks([])
src.utils.set_lims(axs)

In [ ]:
SEL = lambda x: x.mean("month")
# SEL = lambda x : x.sel(month=slice(10,12)).mean("month")

## shared args
kwargs = dict(sel_fn=SEL, norm=False)

fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

## growth rate
plot_timeseries_v2(axs[0], coefs=0.5 * (R - eps), **kwargs)
## Bjerknes
plot_timeseries_v2(axs[1], coefs=R, **kwargs)

## damping
plot_timeseries_v2(axs[2], coefs=alpha, **kwargs)

## difference
plot_timeseries_v2(axs[3], coefs=R + alpha, **kwargs)

## label
labels = [r"$R-\varepsilon$", r"$R$", r"$\alpha$", r"$R_o=R+\alpha$"]
for ax, label in zip(axs, labels):
    ax.set_title(label)

##formatting
# axs[0].set_ylim([-1.75,.5])
# for ax in axs[1:]:
#     ax.set_ylim([-1,3.5])

plt.show()

In [ ]:
SEL = lambda x: x.mean("month")
# SEL = lambda x : x.sel(month=[11,12,1,2]).mean("month")

## shared args
kwargs = dict(sel_fn=SEL, norm=False)

fig, axs = plt.subplots(1, 3, figsize=(7.5, 2.5), layout="constrained")

## F1
plot_timeseries_v2(axs[0], coefs=F1, **kwargs)

## F2
plot_timeseries_v2(axs[1], coefs=F2, **kwargs)

## Product
plot_timeseries_v2(axs[2], coefs=F1 * F2, **kwargs)


## label
labels = ["F1", "F2", "F1*F2"]
for ax, label in zip(axs, labels):
    ax.set_title(label)
plt.show()

In [ ]:
j_var = T_VAR

YEARS = [1870, 2010, 2050, 2080]
colors = cmocean.cm.balance(np.linspace(0.2, 0.8, len(YEARS)))

for param, name in zip(
    [
        R_check - eps,
        R - eps,
        R_check,
        R,
        mu,
        beta * ah,
        gamma.sel(j=j_var),
        alpha,
        (zaf + dd + ndh).sel(j=j_var),
        ekf.sel(j=j_var),
        thf.sel(j=j_var),
        dT_bar * xr.ones_like(thf.sel(j=j_var)),
        w_bar * xr.ones_like(thf.sel(j=j_var)),
    ],
    [
        "R-eps (v1)",
        "R-eps (v2)",
        "R",
        "R (v2)",
        "mu",
        "beta*ah",
        r"$\gamma$",
        r"$\alpha$",
        "zonal",
        "EKF",
        "THF",
        r"$\Delta_z\overline{T}$",
        r"$\overline{w}$",
    ],
):

    print(name)

    fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")

    for ax, n in zip(axs, ["all", "pos", "neg"]):
        ax.set_title(n)
        ax.axhline(0, ls="--", c="k", lw=0.8)
        for i, y in enumerate(YEARS):
            ax.plot(
                R_check.month, param[n].mean("member").sel(year=y), label=y, c=colors[i]
            )

    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])
    axs[0].legend(prop=dict(size=10))

    plt.show()

In [ ]:
## specify feedback
# F = (zaf+dd+ndh)
# F = thf
# F = ekf
# F = R.expand_dims(j=[T_VAR])

get_delta = lambda x: x - x.isel(year=0)
AMP = 0.8
# get_delta = lambda x : x
# AMP = 2

for F, label in zip(
    [
        (R).expand_dims(j=[T_VAR]),
        thf,
        2 * (mu * ah * beta).expand_dims(j=[T_VAR]),
        4 * gamma,
        ekf,
        # zaf+dd+ndh,
        zaf,
        -alpha.expand_dims(j=[T_VAR]),
    ],
    ["R", "thf", "mu*beta*ah", "gamma", "ekf", "zonal", "-alpha"],
):
    print(f"\n\n{label}")
    fig, axs = plt.subplots(1, 3, figsize=(6, 4))

    for ax, n in zip(axs, ["all", "pos", "neg"]):
        ax.contourf(
            zaf.month,
            zaf.year,
            get_delta(F[n].isel(j=0).mean("member")).transpose("year", ...),
            cmap="cmo.balance",
            levels=src.utils.make_cb_range(AMP, AMP / 10),
            extend="both",
        )

        ax.axhline(2010, c="k", ls="--", lw=0.8)
        ax.set_yticks([])
    axs[0].set_yticks([1870, 2010, 2085])
    plt.show()